# Project B - Context-Gap Distillation

KL between the model's predictions with and without the skill document in context, used
as both the importance signal and the distillation loss.

More session-choppable than Project A: each skill-category adapter is self-contained and
`distill/train.py --resume` skips groups already trained. Budget ~5 GPU-hours.

In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'   # cleared only if you prefer a Kaggle Dataset
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

# peft raises rather than degrading when it finds an old torchao, and Kaggle ships
# 0.10.0 against a >0.16 requirement. Nothing here uses torchao, so remove it.
try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



In [ ]:
# re-run this cell (only) if the kernel restarts
import os, sys
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_skills.yaml'
RUNS = '/kaggle/working/artifacts/runs_skill'   # must equal run_root in CFG
SCORES = f'{RUNS}/scores_span.jsonl'
RUN_TOKEN_ARM = True    # second arm: token-granularity gate, see below
RUN_SWEEP = False       # the KL-threshold sweep is a third session, see below

from common.io import load_config
assert load_config(CFG)['run_root'] == RUNS, 'run_root mismatch: distill would not find its scores'
d = load_config(CFG)['distill']
print(f"distill: 3 policies x 3 categories x {d['steps']} steps = {9 * d['steps']} steps")
print('each step runs two forward passes (teacher, student)')
if RUN_TOKEN_ARM:
    print(f"token arm: {9 * d['steps']} more steps, written to a separate report_token/")
if RUN_SWEEP:
    print(f"sweep:   6 configs x 3 categories x {d['steps']} steps = {18 * d['steps']} more")


In [ ]:
# Session persistence: /kaggle/working is wiped when the session restarts.
# To resume across sessions, download the runs.zip this notebook writes at the
# end (right panel -> Output), add it as an input dataset, and set ARCHIVE to it.
ARCHIVE = '/kaggle/input/CHANGE-ME/runs.zip'
import os
if os.path.exists(ARCHIVE):
    run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")
else:
    print(f'no prior archive at {ARCHIVE} - starting fresh (fine for a first run)')


## Gate verification

The riskiest part of the pipeline, and the precondition for everything below it.

`inspect_gate.py` measures what fraction of each demo's **required** strings (the API
identifiers the answer must contain) land inside the gate's selection, and compares that
against a **matched-budget random control**. A gate that does not clear its control cannot
beat the random-span baseline downstream, so read the verdict line before spending GPU on
the distillation.

Span granularity fails this check: mean-pooling KL over a span favours uniformly-surprising
prose over code, where the required identifiers actually live. See
[project_b_findings.md](../docs/project_b_findings.md).

In [ ]:
run(f"python kl_gate/score.py --config {CFG} --granularity span --out {SCORES}")
run(f"python kl_gate/inspect_gate.py --scores {SCORES} --top-frac 0.25 --show 6 --out {RUNS}/gate_report_span.json")


## Distillation and evaluation

`random` is the control that matters: same active-token budget as the KL gate, spans
chosen at random. Any gap between it and `kl_top` is the gate doing real work.

In [ ]:
run(f"python scripts/run_skills.py --config {CFG} --stages distill,eval,report")
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip --prune")
print()
print('=' * 64)
print('DOWNLOAD /kaggle/working/runs.zip NOW (right panel -> Output) or this')
print('run cannot be resumed. Then add it as a dataset and set ARCHIVE.')
print('=' * 64)


## Token-granularity arm

The span gate selects required content *below* a matched-budget random control (-5.1 sigma),
because span-mean KL rewards uniformly-surprising prose and the required identifiers sit in
code as a few spiky tokens among predictable syntax. Scored per token instead, the same KL
signal clears its control (+2.6 sigma at `top_frac=0.25`).

This arm re-scores at token granularity and retrains all three policies. Everything is
written beside the span arm - `scores_token.jsonl`, `distill_*_token_0.25/`, `report_token/`,
`docs/results_skills_token.md` - so the span results are not overwritten.

The question it answers: once the gate genuinely carries signal, does gating beat uniform?
Uniform has coverage 1.0 by construction, so the honest prior is that it still wins and any
gate is a coverage sacrifice that buys nothing at this scale.

In [ ]:
from IPython.display import Image, display
import os
figs = [f'{RUNS}/report/figures/headline_skills.png',
        f'{RUNS}/report_token/figures/headline_skills.png',
        f'{RUNS}/report/figures/kl_threshold_sweep.png']
for path in figs:
    if os.path.exists(path):
        print(path)
        display(Image(path))
    else:
        print(f'missing {path}')
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip")


In [ ]:
if RUN_SWEEP:
    run(f"python scripts/sweep_kl.py --config {CFG} --fracs 0.1,0.25,0.5 --granularities span,token")
    run(f"python eval/skill_report.py --report-dir {RUNS}/report --run-root {RUNS} --out docs/results_skills.md")
else:
    print('sweep skipped. It trains 18 more adapters - roughly triple the main run -')
    print('so give it its own session with RUN_SWEEP = True once the main table looks right.')


In [ ]:
from IPython.display import Image, display
import os
for fig in ['headline_skills.png', 'kl_threshold_sweep.png']:
    path = f'{RUNS}/report/figures/{fig}'
    if os.path.exists(path):
        display(Image(path))
    else:
        print(f'missing {fig}')
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip")
